In [5]:
%pip install be-great pandas sdv scikit-learn numpy plt seaborn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "tabularisai/Qwen3-0.3B-distil"
local_model_path = "./pretrained-qwen3-0.3B"

synthetizer_path="qwen-distil.pickle"
synth_data_output_path="qwen_synth_out.csv"

print(f"Descargando {model_name}...")

# Descargamos Tokenizer y Modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Guardamos en local
tokenizer.save_pretrained(local_model_path)
model.save_pretrained(local_model_path)

print(f"¡Listo! Copia la carpeta '{local_model_path}' a tu máquina sin internet.")

Descargando tabularisai/Qwen3-0.3B-distil...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.89s/it]

¡Listo! Copia la carpeta './pretrained-qwen3-0.3B' a tu máquina sin internet.


In [7]:
import pandas as pd
import numpy as np

from be_great import GReaT
from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality

# 1. Cargar el dataset adjunto
df_real = pd.read_csv('../../datasources/bank_full_dataset/bank-full.csv', sep=';')

# Para la PoC, tomamos una muestra representativa para optimizar tiempos de GPU/CPU
# En producción (Vertex AI), usaríamos el dataset completo.
df_train = df_real.sample(n=1000, random_state=42)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_real) 

print(f"Dataset cargado: {df_real.shape[0]} registros.")
print("Columnas detectadas:", df_real.columns.tolist())

num_cols = df_real.select_dtypes(include=[np.number]).columns
print("Columnas numericas:", num_cols)
cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
print("Columnas categoricas:", cat_cols)

Dataset cargado: 45211 registros.
Columnas detectadas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Columnas numericas: Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')
Columnas categoricas: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')


In [8]:
import pandas as pd
from be_great import GReaT

# 1. Ruta donde copiaste la carpeta

# 3. Inicializamos GReaT apuntando a la ruta local
# Importante: llm=local_model_path evita que intente conectar a internet
model = GReaT(
    llm=local_model_path,  
    float_precision=2,  # Limit floating-point precision to 3 decimal places
    batch_size=16,     # Use smaller batch size for small datasets
    epochs=16,          # Train for more epochs with small data
    bf16=True           # Enable half-precision training for faster computation and lower memory usage
)

# Entrenamiento
model.fit(df_train)
model.save(synthetizer_path)


Loading weights: 100%|██████████| 156/156 [00:00<00:00, 2271.47it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
500,0.907354
1000,0.814098


Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.79s/it]


In [9]:
# 3. Generación de datos sintéticos
# Vamos a generar 1000 registros nuevos "soñados" por el modelo
df_synthetic = model.sample(n_samples=1000, guided_sampling=True)

# Limpieza rápida: GREAT a veces genera strings con espacios extra
df_synthetic = df_synthetic.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("Muestra de datos sintéticos generados:")
display(df_synthetic.head())

df_synthetic.to_csv(synth_data_output_path)

100%|██████████| 1000/1000 [1:57:20<00:00,  7.04s/it]


Muestra de datos sintéticos generados:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,40,blue-collar,married,secondary,no,239,yes,no,cellular,20,may,484,3,262,2,success,no
1,39,technician,single,secondary,no,56,yes,no,unknown,18,may,325,1,-1,0,unknown,no
2,32,management,married,tertiary,no,1258,no,no,cellular,3,aug,169,2,-1,0,unknown,no
3,28,management,single,tertiary,no,359,yes,no,unknown,12,may,1061,2,-1,0,unknown,no
4,33,blue-collar,single,secondary,no,-253,yes,no,unknown,20,may,192,1,-1,0,unknown,no


In [10]:
# 5. Función de reporte de calidad
def reporte_calidad(real, synth):
    print("\n" + "="*40)
    print("   REPORTE DE FIDELIDAD ESTADÍSTICA")
    print("="*40 + "\n")
      
    # Test de Proporciones para Categóricas (Diferencia Media)
    cat_cols = real.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        f_real = real[col].value_counts(normalize=True).sort_index()
        f_synth = synth[col].value_counts(normalize=True).sort_index()
        diff = (f_real - f_synth).abs().mean()
        status = "✅ FIEL" if diff < 0.05 else "❌ DESVIADO"
        print(f"Prop. Diff [{col:10}]: {diff:.4f} {status}")

reporte_calidad(df_real, df_synthetic)


   REPORTE DE FIDELIDAD ESTADÍSTICA

Prop. Diff [job       ]: 0.0252 ✅ FIEL
Prop. Diff [marital   ]: 0.0374 ✅ FIEL
Prop. Diff [education ]: 0.0408 ✅ FIEL
Prop. Diff [default   ]: 0.0170 ✅ FIEL
Prop. Diff [housing   ]: 0.0378 ✅ FIEL
Prop. Diff [loan      ]: 0.1052 ❌ DESVIADO
Prop. Diff [contact   ]: 0.1040 ❌ DESVIADO
Prop. Diff [month     ]: 0.0229 ✅ FIEL
Prop. Diff [poutcome  ]: 0.0508 ❌ DESVIADO
Prop. Diff [y         ]: 0.0670 ❌ DESVIADO


In [11]:
import torch
import gc

# 1. Eliminar el objeto del modelo (y cualquier otro pesado)
if 'model' in locals():
    del model
if 'df_synthetic' in locals():
    del df_synthetic

# 2. Forzar la recolección de basura de Python
gc.collect()

# 3. Vaciar el caché de PyTorch (esto libera la VRAM físicamente)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
print("Memoria de GPU liberada.")

Memoria de GPU liberada.
